| **A. Clean Data** | B. Tidy Data | C. Find, Filter, Group | D. Charts and Graphs |
|---|---|---|---|

# The Beatles, Part A: Clean Data

Cleaning data is an important precursor to analyzing data, and often represents the biggest part of a data analysis project. In this notebook we take a single, rich (and messy!) dataset about the Beatles and prepare it for analysis. At the end we save a **clean** version as a Pickle (`.pkl`) file that the next three notebooks will build on.

Resources: [Official Pandas Docs](https://pandas.pydata.org/docs/) · [W3Schools Pandas](https://www.w3schools.com/python/pandas/default.asp) · [Pandas Cheat Sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) · [Pandas string methods](https://pandas.pydata.org/docs/user_guide/text.html#method-summary)

| | Contents of this Notebook |
|---|---|
| 1. | Meet the Data |
| 2. | Tidy Column Names |
| 3. | Missing Data: Find, Decide, Fill or Drop |
| 4. | Wrong or Inconsistent Values |
| 5. | Using a Dictionary with `map()` |
| 6. | Cleaning with Functions and `apply()` |
| 7. | Wrong Data Types |
| 8. | Duplicates |
| 9. | Save the Clean Data as a Pickle |
| 10. | Best Practice: One Reproducible Cleaning Function |

## 1. Meet the Data

Our file, `Beatles_Belgrade_Complete.csv`, combines two sources that you may have met separately before:

* **Spotify** audio features for each track: `Popularity`, `Duration`, `Key`, `Mode`, `Tempo`, and the 0–1 ratings `Valence`, `Danceability`, `Energy`, `Acousticness`, `Instrumentalness`, `Liveness`, `Speechiness`.
* **University of Belgrade** metadata: year, album debut, songwriter(s), lead vocal(s), whether the song is a cover, plus
  * **categorical labels** for `Genre`, `Styles`, `Themes`, and `Moods` (several labels per song!)
  * **chart and ranking data**: Billboard, UK charts (via Wikipedia and *The Guardian*), and several "Top 50 Beatles songs" lists (Rolling Stone, NME, *USA Today*, Vulture, and others).

Because the merging has already been done, we can go straight to cleaning.

### Research questions to keep in mind

Cleaning decisions always depend on what we want to ask. Across these four notebooks we will pursue questions like:

* How did the Beatles' **sound change over time** (energy, acousticness, valence by year or album)?
* Do **Lennon, McCartney, and Harrison** songs differ in genre, mood, or audio features?
* Does historical **chart success** (Billboard) relate to present-day **Spotify popularity**?
* Which **genres, styles, themes, and moods** are most typical of the band, and how do they cluster?
* How do **covers** differ from **original** Beatles songs?

In [12]:
import pandas as pd

# show all columns when we display a DataFrame
pd.set_option('display.max_columns', None)

### Load the CSV

The CSV sits in the same folder as this notebook, so we can load it with a simple relative path.

In [13]:
beatles_raw = pd.read_csv('Beatles_Belgrade_Complete.csv')
beatles_raw.shape

(285, 45)

In [14]:
beatles_raw.head(3)

,URI,Title,Year,Album,Popularity,Duration,Key,Mode,Tempo,Time_signature,Valence,Danceability,Energy,Loudness,Acousticness,Instrumentalness,Liveness,Speechiness,Album debut,Other releases,Single A side,Single B side,Single certification,Genre,Styles,Themes,Moods,Songwriter(s),Lead vocal(s),Cover,Covered by,Chart position UK (Wikipedia),Chart position US (Wikipedia),Highest position (The Guardian),Weeks on chart in UK (The Guardian),Weeks at No1 in UK (The Guardian),Highest position (Billboard),Weeks at No1 (Billboard),Top 50 (Billboard),Top 50 (Ultimate classic rock),Top 50 (Rolling Stone),Top 50 (NME),Top 50 (Top50songs.org),"Top 50 (USA today, 2017)","Top 50 (Vulture, by Bill Wyman)"
0,spotify:track:2FDEHIMkjxFLzj688M2I3h,(You're So Square) Baby I Don't Care - Studio Jam,NaN,The Beatles,29.0,43.0,9.0,1.0,112.173,4.0,0.353,0.411,0.840,-9.919,0.09910,0.000,0.364,0.0896,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,spotify:track:2HvTGx5fzFGpHSyRNvXd9T,12-bar Original,1965,Anthology 2,31.0,175.0,9.0,1.0,122.678,4.0,0.566,0.600,0.535,-8.178,0.00154,0.102,0.123,0.0298,Anthology 2,NaN,NaN,NaN,NaN,Blues,NaN,NaN,Passionate,"Lennon, McCartney, Harrison and Starkey",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,spotify:track:34dsKRJHIadrrNdCDtMwGn,A Beginning - Anthology 3 Version,NaN,Anthology 3,29.0,50.0,0.0,1.0,90.588,3.0,0.033,0.264,0.293,-16.498,0.95800,0.942,0.338,0.0377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


`info()` shows each column's **data type** and the number of **non-null** (not missing) values. Already we can see problems: `Year` is stored as text (`object`/`str`), not a number, and many columns have far fewer than 285 values.

In [15]:
beatles_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285 entries, 0 to 284
Data columns (total 45 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   URI                                  285 non-null    object 
 1   Title                                285 non-null    object 
 2   Year                                 277 non-null    object 
 3   Album                                284 non-null    object 
 4   Popularity                           285 non-null    float64
 5   Duration                             285 non-null    float64
 6   Key                                  285 non-null    float64
 7   Mode                                 285 non-null    float64
 8   Tempo                                285 non-null    float64
 9   Time_signature                       285 non-null    float64
 10  Valence                              285 non-null    float64
 11  Danceability                    

## 2. Tidy Column Names

Column names like `Songwriter(s)` or `Top 50 (USA today, 2017)` are awkward to type, and spaces and parentheses cause trouble in some Pandas methods. A common convention is **snake_case**: all lowercase, words joined by underscores.

Rather than renaming 45 columns by hand, we write a small **function** and apply it to every column name. This is the same approach we will use for cleaning values below.

In [16]:
def clean_column_name(name):
    name = name.lower()
    name = name.replace('(s)', '')           # 'songwriter(s)' -> 'songwriter'
    for character in ['(', ')', ',', '.']:   # remove punctuation
        name = name.replace(character, '')
    name = name.strip().replace(' ', '_')    # spaces -> underscores
    return name

# always work on a copy, so the raw data stays untouched
beatles = beatles_raw.copy()
beatles.columns = [clean_column_name(col) for col in beatles.columns]
list(beatles.columns)

['uri',
 'title',
 'year',
 'album',
 'popularity',
 'duration',
 'key',
 'mode',
 'tempo',
 'time_signature',
 'valence',
 'danceability',
 'energy',
 'loudness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'speechiness',
 'album_debut',
 'other_releases',
 'single_a_side',
 'single_b_side',
 'single_certification',
 'genre',
 'styles',
 'themes',
 'moods',
 'songwriter',
 'lead_vocal',
 'cover',
 'covered_by',
 'chart_position_uk_wikipedia',
 'chart_position_us_wikipedia',
 'highest_position_the_guardian',
 'weeks_on_chart_in_uk_the_guardian',
 'weeks_at_no1_in_uk_the_guardian',
 'highest_position_billboard',
 'weeks_at_no1_billboard',
 'top_50_billboard',
 'top_50_ultimate_classic_rock',
 'top_50_rolling_stone',
 'top_50_nme',
 'top_50_top50songsorg',
 'top_50_usa_today_2017',
 'top_50_vulture_by_bill_wyman']

## 3. Missing Data: Find, Decide, Fill or Drop

Missing data (`NaN`, `None`, `pd.NA`) is one of the most common problems in real datasets. String methods and many calculations will fail or give odd results when they meet a missing value, so we must **find** missing values and then **decide** what they mean before we **fill** or **drop** them.

### Find: how much is missing, and where?

In [6]:
missing = pd.DataFrame({
    'missing': beatles.isna().sum(),
    'percent': (beatles.isna().mean() * 100).round(1)
})
missing[missing['missing'] > 0].sort_values('missing', ascending=False)

,missing,percent
weeks_at_no1_billboard,270,94.7
weeks_at_no1_in_uk_the_guardian,269,94.4
highest_position_the_guardian,254,89.1
weeks_on_chart_in_uk_the_guardian,254,89.1
top_50_vulture_by_bill_wyman,243,85.3
chart_position_uk_wikipedia,240,84.2
top_50_nme,236,82.8
top_50_rolling_stone,236,82.8
top_50_billboard,236,82.8
highest_position_billboard,236,82.8


### Decide: not all missing data means the same thing

Reading down that table, we can see **three very different kinds** of missing data:

1. **Missing because the event never happened.** The chart and ranking columns (`top_50_billboard`, `chart_position_us_wikipedia`, `weeks_at_no1_billboard`, ...) are empty for most songs because most songs were never hits. `NaN` here is *information*: "did not chart." We should **not** fill these with `0`: a chart position of 0 would be meaningless (or would look like "better than No. 1"!). We will leave them as `NaN` and create a simple True/False column instead.
2. **Missing optional labels.** `themes`, `styles`, `moods`, `lead_vocal`, `album_debut`: some songs simply were not labelled. We can fill these with a sensible default.
3. **Whole rows with no Belgrade metadata.** Some Spotify tracks (studio jams, alternate takes) have no genre, songwriter, or year at all.

Let's look at the third group first. Here is a **filter** (more on these in Notebook C) showing rows with no `genre`:

In [17]:
selected_cols = ['title', 'year', 'album', 'songwriter', 'genre', 'moods']

beatles[beatles['genre'].isna()][selected_cols]

,title,year,album,songwriter,genre,moods
0,(You're So Square) Baby I Don't Care - Studio Jam,NaN,The Beatles,NaN,NaN,NaN
2,A Beginning - Anthology 3 Version,NaN,Anthology 3,NaN,NaN,NaN
8,Across the Universe - Take 2 / Anthology 2 Ver...,NaN,Anthology 2,NaN,NaN,NaN
30,Besame Mucho,NaN,Anthology 1,NaN,NaN,NaN
35,Can You Take Me Back? - Take 1,NaN,The Beatles,NaN,NaN,NaN
83,"Hallelujah, I Love Her So",1960,Anthology 1,NaN,NaN,NaN
152,Los Paranoias - Studio Jam,NaN,The Beatles,NaN,NaN,NaN


### Drop: rows we cannot use

These seven tracks are outtakes and jams that exist *only in the Spotify data*. With no genre, songwriter, or year they cannot take part in any of our research questions, so we **drop** them. `dropna(subset=[...])` drops rows that are missing values *in the listed columns only*.

> Dropping data is a research decision. Write down *why* you dropped it (as we do here), so others can evaluate your choice.

In [18]:
beatles = beatles.dropna(subset=['genre'])
beatles.shape

(278, 45)

### Check again: what is still missing in the descriptive columns?

In [19]:
descriptive_cols = ['year', 'album', 'album_debut', 'songwriter', 'lead_vocal',
                    'genre', 'styles', 'themes', 'moods']
beatles[descriptive_cols].isna().sum()

year            2
album           1
album_debut     3
songwriter      0
lead_vocal      5
genre           0
styles         21
themes         94
moods          18
dtype: int64

Let's look at the songs with no `lead_vocal` to see what they might have in common.

In [20]:
selected_cols = ['title', 'year', 'songwriter', 'lead_vocal', 'instrumentalness']

beatles[beatles['lead_vocal'].isna()][selected_cols]

,title,year,songwriter,lead_vocal,instrumentalness
1,12-bar Original,1965,"Lennon, McCartney, Harrison and Starkey",NaN,0.102000
39,Cayenne,1960,McCartney,NaN,0.895000
46,Cry for a Shadow,1961,Lennon and Harrison,NaN,0.000154
67,Flying,1967,"Lennon, McCartney, Harrison and Starkey",NaN,0.880000
197,Revolution 9,1968,"Lennon, with Ono and Harrison",NaN,0.118000


Spotify audio features seem to confirm that some are **instrumentals** (or, in the case of "Revolution 9", a sound collage). We should check these to be sure but `NaN` here might really mean "no lead vocal", and we can fill it accordingly.  

### Fill: one column at a time, with a value that makes sense

`fillna()` replaces missing values. Because each column has a different meaning, we fill them one at a time:

* `lead_vocal` → `'None (instrumental)'`
* `album` and `album_debut` → `'Unreleased'` (these are demos and songs the Beatles gave to other artists)
* `styles`, `themes`, `moods` → an **empty string** `''`. This keeps the column as text, so string methods like `.str.contains()` will work later without errors.
* `year` → we will leave the two missing years missing, but store them properly as a *nullable integer* (see Section 7).

In [22]:
beatles['lead_vocal'] = beatles['lead_vocal'].fillna('None (instrumental)')
beatles['album'] = beatles['album'].fillna('Unreleased')
beatles['album_debut'] = beatles['album_debut'].fillna('Unreleased')

for col in ['styles', 'themes', 'moods']:
    beatles[col] = beatles[col].fillna('')


# check the NA values again
beatles[descriptive_cols].isna().sum()

year           2
album          0
album_debut    0
songwriter     0
lead_vocal     0
genre          0
styles         0
themes         0
moods          0
dtype: int64

### Missing as information: a True/False "charted" column

Instead of filling the chart columns, we create **Boolean** (True/False) columns that record whether a value exists. `.notna()` returns `True` where there *is* a value.

The `cover` column works the same way: it holds `'Y'` for covers and `NaN` for Beatles originals, so it is really a True/False column in disguise.

In [11]:
beatles['billboard_top50'] = beatles['top_50_billboard'].notna()
beatles['is_cover'] = beatles['cover'] == 'Y'
beatles = beatles.drop(columns=['cover'])

beatles[['billboard_top50', 'is_cover']].value_counts()

billboard_top50  is_cover
False            False       168
                 True         61
True             False        44
                 True          5
Name: count, dtype: int64

## 4. Wrong or Inconsistent Values

A quick way to spot inconsistencies is to look at the **unique values** in a column, sorted, or with their counts via `value_counts()`.

### Album names

In [13]:
beatles['album'].value_counts()

album
The Beatles                                31
Live at the BBC                            27
Past Masters (Vols. 1 & 2 / Remastered)    26
Anthology 1                                19
Abbey Road                                 17
Please Please Me                           14
Help!                                      14
With the Beatles                           14
Revolver                                   14
Beatles for Sale                           14
Rubber Soul                                14
Sgt. Pepper's Lonely Hearts Club Band      13
A Hard Day's Night                         13
Let It Be                                  12
Magical Mystery Tour                       11
Anthology 3                                 9
Anthology 2                                 4
Yellow Submarine                            4
Live At The BBC                             3
On Air – Live at the BBC Volume 2           1
Abbey Road (Super Deluxe Edition)           1
On Air - Live At The BBC (Vo

Notice the problems:

* `Live at the BBC` and `Live At The BBC` (capitalization)
* `On Air – Live at the BBC Volume 2` and `On Air - Live At The BBC (Vol.2)` (different dashes and wording)
* `Abbey Road (Super Deluxe Edition)` is a reissue of `Abbey Road`

We fix these with a **dictionary** in the next section. First, let's look at text columns that hold *lists* of labels.

### Years

`year` was read as text because two entries contain *two* years (a John Lennon demo, later completed by the other Beatles):

In [14]:
beatles[beatles['year'].str.contains('/', na=False)][['title', 'year', 'album']]

,title,year,album
70,Free As a Bird,1977/1994,Anthology 1
194,Real Love,1980/1995,Anthology 2


We keep the year the **Beatles** recorded the song (the second year). `.str.split('/')` turns `'1977/1994'` into the list `['1977', '1994']`, and `.str[-1]` takes the last item. Years without a slash are unchanged.

In [15]:
beatles['year'] = beatles['year'].str.split('/').str[-1]
beatles['year'].unique()

array(['1965', '1967', '1964', '1963', '1968', '1961', '1969', '1966',
       '1962', '1960', '1994', nan, '1970', '1958', '1995'], dtype=object)

### Song titles

A few titles carry version information from Spotify, like `Circles - Esher Demo`. Since all our other data is about the *song*, we keep just the part before `' - '`:

In [16]:
beatles[beatles['title'].str.contains(' - ')]['title']

41                                  Circles - Esher Demo
219                           Sour Milk Sea - Esher Demo
237    The Hippy Hippy Shake - Live at the Bbc for "p...
Name: title, dtype: str

In [17]:
beatles['title'] = beatles['title'].str.split(' - ').str[0].str.strip()
beatles[beatles['title'].str.contains('Circles|Sour Milk|Hippy')]['title']

41                   Circles
219            Sour Milk Sea
237    The Hippy Hippy Shake
Name: title, dtype: object

### Categorical labels: Genre, Styles, Themes, Moods

These four columns each hold **several labels in one cell**. Three use commas, but `moods` uses spaces:

In [18]:
beatles[['title', 'genre', 'styles', 'themes', 'moods']].head(3)

,title,genre,styles,themes,moods
1,12-bar Original,Blues,,,Passionate
3,A Day in the Life,"Psychedelic Rock, Art Rock, Pop/Rock","British Psychedelia, Contemporary Pop/Rock, Ea...",,Ambitious Complex Dramatic Dreamy Eerie Elabor...
4,A Hard Day's Night,"Rock, Electronic, Pop/Rock","British Invasion, Contemporary Pop/Rock, Early...","Everyday Life, In Love, Relationships",Carefree Cheerful Explosive Marching Optimisti...


To find inconsistencies inside these lists, we temporarily split and **explode** them (you'll learn this properly in Notebook B), then count the individual labels. Here are all the genre labels:

In [19]:
beatles['genre'].str.split(',').explode().str.strip().value_counts().sort_index()

genre
Acid Rock          2
Acid Rock[         1
Art Pop            2
Art Rock           3
Avant-Garde        2
                  ..
Stage&Screen       1
Sunshine Pop       1
Swamp Pop          1
Symphonic Rock     1
Vaudeville Rock    1
Name: count, Length: 83, dtype: int64

Scan that list carefully and you'll find several problems:

* **Stray characters**: `Heavy Metal[` and `Acid Rock[` (left over from Wikipedia footnote brackets)
* **Capitalization**: `Hard rock` / `Hard Rock`, `Psychedelic folk` / `Psychedelic Folk`, `Rock and roll` / `Rock and Roll`, ...
* **Same idea, different spelling**: `Rock & Roll` / `Rock and Roll`; `Pop Rock` / `Pop/Rock`; `Stage&Screen`
* **Redundant words**: `Children's Music` / `Children's`, `Experimental Music` / `Experimental` (having "music" in a list of music genres is not very useful)
* **A missing comma**: `FolkPop/Rock` should be two labels, `Folk` and `Pop/Rock`

The `styles` column has typos of its own:

In [20]:
problem_styles = beatles['styles'].str.contains(r'\.|RockHard|Pop/Roc,', regex=True)
beatles[problem_styles][['title', 'styles']]

,title,styles
16,And I Love Her,"British Invasion. Contemporary Pop/Rock, Early..."
192,Polythene Pam,"Album Rock, Contemporary Pop/Roc,k, Early Pop/..."
235,The End,"Album Rock, Contemporary Pop/Rock, Early Pop/R..."


Here, `.str.replace()` is the right tool: each fix is a small, targeted text substitution.

* `'British Invasion. Contemporary'` → a period where a comma should be
* `'Prog-RockHard Rock'` → a missing comma
* `'Pop/Roc,k'` → a comma in the wrong place

We fix the missing comma in `genre` (`FolkPop/Rock`) the same way.

In [21]:
beatles['styles'] = (beatles['styles']
                     .str.replace('. ', ', ', regex=False)
                     .str.replace('Prog-RockHard Rock', 'Prog-Rock, Hard Rock', regex=False)
                     .str.replace(r'Pop/Roc,\s*k', 'Pop/Rock', regex=True))

beatles['genre'] = beatles['genre'].str.replace('FolkPop/Rock', 'Folk, Pop/Rock', regex=False)

beatles.loc[problem_styles, ['title', 'styles']]

,title,styles
16,And I Love Her,"British Invasion, Contemporary Pop/Rock, Early..."
192,Polythene Pam,"Album Rock, Contemporary Pop/Rock, Early Pop/R..."
235,The End,"Album Rock, Contemporary Pop/Rock, Early Pop/R..."


### Moods: make the separator consistent

`moods` uses spaces between labels. Luckily, no single mood contains a space (multi-word moods are joined with `/`, as in `Laid-Back/Mellow`), so we can safely convert it to the same comma-separated format as the others. `.str.split()` with no argument splits on any whitespace; `', '.join` puts the list back together with commas.

In [22]:
beatles['moods'] = beatles['moods'].str.split().str.join(', ')
beatles['moods'].head(3)

1                                           Passionate
3    Ambitious, Complex, Dramatic, Dreamy, Eerie, E...
4    Carefree, Cheerful, Explosive, Marching, Optim...
Name: moods, dtype: object

## 5. Using a Dictionary with `map()` or `replace()`

When you know exactly which values should become which, a **dictionary** is the clearest tool. The *keys* are the old values; the *values* are the new ones.

A handy trick: `dict.fromkeys(df['column'].unique())` builds a dictionary with every unique value as a key, ready for you to fill in.

In [23]:
dict.fromkeys(beatles['album'].unique())

{'Anthology 2': None,
 "Sgt. Pepper's Lonely Hearts Club Band": None,
 "A Hard Day's Night": None,
 'Live at the BBC': None,
 'Please Please Me': None,
 'Let It Be': None,
 'Help!': None,
 'Anthology 1': None,
 'With the Beatles': None,
 'Anthology 3': None,
 'Yellow Submarine': None,
 'Magical Mystery Tour': None,
 'Revolver': None,
 'Beatles for Sale': None,
 'The Beatles': None,
 'Past Masters (Vols. 1 & 2 / Remastered)': None,
 'On Air – Live at the BBC Volume 2': None,
 'Abbey Road': None,
 'Rubber Soul': None,
 'Abbey Road (Super Deluxe Edition)': None,
 'Live At The BBC': None,
 'On Air - Live At The BBC (Vol.2)': None,
 "Rock 'n' Roll Music": None,
 'Unreleased': None}

We only need to list the entries we want to change. With `.replace(dictionary)`, anything *not* in the dictionary stays as it is.

> `.map(dictionary)` is similar, but it returns `NaN` for any value that is not a key. That is useful when you *want* to map every value (as we'll do for musical keys below), but risky otherwise.

In [24]:
album_fixes = {
    'Live At The BBC': 'Live at the BBC',
    'On Air - Live At The BBC (Vol.2)': 'On Air – Live at the BBC Volume 2',
    'Abbey Road (Super Deluxe Edition)': 'Abbey Road',
}
beatles['album'] = beatles['album'].replace(album_fixes)
beatles['album'].value_counts()

album
The Beatles                                31
Live at the BBC                            30
Past Masters (Vols. 1 & 2 / Remastered)    26
Anthology 1                                19
Abbey Road                                 18
Please Please Me                           14
Help!                                      14
With the Beatles                           14
Revolver                                   14
Beatles for Sale                           14
Rubber Soul                                14
Sgt. Pepper's Lonely Hearts Club Band      13
A Hard Day's Night                         13
Let It Be                                  12
Magical Mystery Tour                       11
Anthology 3                                 9
Anthology 2                                 4
Yellow Submarine                            4
On Air – Live at the BBC Volume 2           2
Rock 'n' Roll Music                         1
Unreleased                                  1
Name: count, dtype: int64

### Mapping codes to meaningful labels

Spotify stores `key` as a number (0 = C, 1 = C♯/D♭, ..., 11 = B) and `mode` as 1 (major) or 0 (minor). Numbers are fine for a computer, but readable labels are better for grouping and charting. Here we **do** want every value translated, so `.map()` is the right choice. We keep the original number columns too.

In [25]:
key_names = {0: 'C', 1: 'C#/Db', 2: 'D', 3: 'D#/Eb', 4: 'E', 5: 'F',
             6: 'F#/Gb', 7: 'G', 8: 'G#/Ab', 9: 'A', 10: 'A#/Bb', 11: 'B'}
mode_names = {1: 'major', 0: 'minor'}

beatles['key_name'] = beatles['key'].map(key_names)
beatles['mode_name'] = beatles['mode'].map(mode_names)

beatles[['title', 'key', 'key_name', 'mode', 'mode_name']].head()

,title,key,key_name,mode,mode_name
1,12-bar Original,9.0,A,1.0,major
3,A Day in the Life,4.0,E,0.0,minor
4,A Hard Day's Night,0.0,C,1.0,major
5,A Shot of Rhythm and Blues,2.0,D,1.0,major
6,A Taste of Honey,1.0,C#/Db,0.0,minor


## 6. Cleaning with Functions and `apply()`

When the fix needs *logic* (not just a lookup), write a **function** and pass it to `.apply()`, which runs it on every value in the column.

### Cleaning a list of labels

Our genre problems live *inside* comma-separated lists, so we write a function that:

1. splits the text into individual labels,
2. strips spaces and stray `[` characters,
3. fixes spelling variants using a dictionary,
4. removes duplicates (while keeping the order), and
5. joins the labels back into one string.

We can use the same function for `genre`, `styles`, and `themes`.

In [26]:
label_fixes = {
    # capitalization
    'Hard rock': 'Hard Rock',
    'Country rock': 'Country Rock',
    'Jangle pop': 'Jangle Pop',
    'Psychedelic folk': 'Psychedelic Folk',
    'Psychedelic pop': 'Psychedelic Pop',
    'Experimental pop': 'Experimental Pop',
    'Folk blues': 'Folk Blues',
    'Rock and roll': 'Rock and Roll',
    # same idea, different spelling
    'Rock & Roll': 'Rock and Roll',
    'Pop Rock': 'Pop/Rock',
    'Stage&Screen': 'Stage and Screen',
    # redundant words
    "Children's Music": "Children's",
    'Experimental Music': 'Experimental',
}

def clean_label_list(text):
    labels = []
    for label in text.split(','):
        label = label.strip().strip('[').strip()
        label = label_fixes.get(label, label)   # look up a fix; if none, keep the label
        if label and label not in labels:
            labels.append(label)
    return ', '.join(labels)

# test the function on one messy example before applying it to a whole column
clean_label_list('Hard Rock, Heavy Metal[, Proto-Punk, Pop/Rock, Hard rock')

'Hard Rock, Heavy Metal, Proto-Punk, Pop/Rock'

In [27]:
for col in ['genre', 'styles', 'themes', 'moods']:
    beatles[col] = beatles[col].apply(clean_label_list)

# check the result: no more duplicates by capitalization or stray brackets
beatles['genre'].str.split(', ').explode().value_counts().sort_index()

genre
Acid Rock           3
Art Pop             2
Art Rock            3
Avant-Garde         2
Avant-Pop           1
                   ..
Stage and Screen    1
Sunshine Pop        1
Swamp Pop           1
Symphonic Rock      1
Vaudeville Rock     1
Name: count, Length: 70, dtype: int64

> **Note on "Rock & Roll"**: in `genre` we merged `Rock & Roll` into `Rock and Roll`. But in `styles`, `Rock & Roll` is the *only* spelling, so our fix renames it there too. That's fine: consistency across columns is a bonus. Always check whether a dictionary of fixes has unintended effects on other columns!

### Simplifying a complicated column: the principal songwriter

The `songwriter` column is very detailed: `McCartney, with Lennon`, `Lennon and McCartney`, `Blackwell, Johnson, Penniman`, ...

In [28]:
beatles['songwriter'].value_counts().head(15)

songwriter
McCartney                                       61
Lennon                                          59
Harrison                                        26
McCartney, with Lennon                          20
Lennon and McCartney                            16
Lennon, with McCartney                          13
Berry                                            9
Williams                                         3
Goffin, King                                     3
Leiber, Stoller                                  3
Lennon, McCartney, Harrison and Starkey          2
Lennon, with McCartney, Harrison and Starkey     2
Perkins                                          2
Thompson                                         1
Scott, Marlow                                    1
Name: count, dtype: int64

For our research questions ("Do Lennon and McCartney songs differ?") we need a simpler **category**. We'll write a function that returns:

* the Beatle named **before** any `with` (the *principal* writer): `'Lennon'`, `'McCartney'`, `'Harrison'`, or `'Starkey'`
* `'Lennon & McCartney'` when both are credited equally (e.g. `Lennon and McCartney`)
* `'Whole Band'` when three or more Beatles are credited equally
* `'Cover'` when no Beatle wrote the song (or it is a traditional tune)

We keep the detailed column and add our simplified one as `songwriter_group`.

In [29]:
beatle_names = ['Lennon', 'McCartney', 'Harrison', 'Starkey']

def songwriter_group(credit):
    if 'Traditional' in credit:
        return 'Cover'
    principal = credit.split('with')[0]               # ignore everyone after 'with'
    writers = [name for name in beatle_names if name in principal]
    if len(writers) == 0:
        return 'Cover'
    if len(writers) == 1:
        return writers[0]
    if len(writers) == 2:
        return ' & '.join(writers)
    return 'Whole Band'

beatles['songwriter_group'] = beatles['songwriter'].apply(songwriter_group)
beatles['songwriter_group'].value_counts()

songwriter_group
McCartney               81
Lennon                  77
Cover                   69
Harrison                27
Lennon & McCartney      17
Whole Band               3
Starkey                  2
Lennon & Harrison        1
McCartney & Harrison     1
Name: count, dtype: int64

Let's check a few tricky cases side by side:

In [30]:
beatles[['songwriter', 'songwriter_group']].drop_duplicates().sample(12, random_state=3)

,songwriter,songwriter_group
145,"Twomey, Wise, Weisman",Cover
10,"Yellen, Ager",Cover
180,"Starkey, with uncredited assistance from Harrison",Starkey
149,"J. Burnette, D. Burnette, Burlison, Mortimer",Cover
13,Harrison,Harrison
26,Williams,Cover
134,McCartney and Harrison,McCartney & Harrison
190,"Dobbins, Garrett, Holland, Bateman, Gorman",Cover
102,Stan Kesler and Charlie Feathers,Cover
250,Willson,Cover


## 7. Wrong Data Types

Pandas guesses data types when it reads a CSV, and it often guesses imperfectly:

* `year` is text. We want numbers, so we can sort, compare (`year > 1965`), and plot.
* `key`, `mode`, `time_signature`, `duration`, `popularity` are floats (`43.0`), but they are really whole numbers.

We use `pd.to_numeric()` and `.astype()`. Note the capital **`'Int64'`**: this is Pandas' *nullable integer* type, which allows whole numbers **and** missing values (`<NA>`). The ordinary `'int64'` would fail on our two missing years.

In [31]:
beatles['year'] = pd.to_numeric(beatles['year']).astype('Int64')

int_cols = ['popularity', 'duration', 'key', 'mode', 'time_signature', 'other_releases', 'covered_by']
for col in int_cols:
    beatles[col] = beatles[col].astype('Int64')

beatles[['year'] + int_cols].dtypes

year              Int64
popularity        Int64
duration          Int64
key               Int64
mode              Int64
time_signature    Int64
other_releases    Int64
covered_by        Int64
dtype: object

> Pandas also has a powerful **datetime** type (`pd.to_datetime(beatles['year'].astype(str), format='%Y')`). It is essential when you have full dates (day-month-year), but for years alone a plain integer is simpler and works perfectly for sorting, filtering, and charts.

A quick sanity check: are all years plausible?

In [32]:
beatles['year'].describe()

count          276.0
mean     1965.652174
std         3.453909
min           1958.0
25%           1963.0
50%           1965.0
75%           1968.0
max           1995.0
Name: year, dtype: Float64

## 8. Duplicates

Duplicate rows inflate counts and distort averages. `.duplicated()` finds rows that are identical to an earlier row. Since we shortened some titles, let's also check for duplicate **titles**.

In [33]:
print('duplicate rows:  ', beatles.duplicated().sum())
print('duplicate titles:', beatles.duplicated(subset=['title']).sum())

duplicate rows:   0
duplicate titles: 0


No duplicates. If there were, `beatles.drop_duplicates()` (or `beatles.drop_duplicates(subset=['title'])`) would remove them.

## 9. Save the Clean Data as a Pickle

First, let's put the columns in a logical order: identifiers, Beatles metadata, categorical labels, Spotify audio features, then chart data. We also reset the index, since we dropped rows earlier.

In [34]:
id_cols       = ['title', 'year', 'album', 'album_debut', 'songwriter', 'songwriter_group',
                 'lead_vocal', 'is_cover']
label_cols    = ['genre', 'styles', 'themes', 'moods']
spotify_cols  = ['popularity', 'duration', 'key', 'key_name', 'mode', 'mode_name', 'tempo',
                 'time_signature', 'valence', 'danceability', 'energy', 'loudness',
                 'acousticness', 'instrumentalness', 'liveness', 'speechiness']
release_cols  = ['other_releases', 'single_a_side', 'single_b_side', 'single_certification', 'covered_by']
chart_cols    = [col for col in beatles.columns if col.startswith(('chart_', 'highest_', 'weeks_', 'top_50'))]
chart_cols    = ['billboard_top50'] + chart_cols

beatles_clean = beatles[id_cols + label_cols + spotify_cols + release_cols + chart_cols + ['uri']]
beatles_clean = beatles_clean.reset_index(drop=True)

# confirm we kept every column exactly once
print(len(beatles_clean.columns), 'columns (was', len(beatles.columns), ')')
beatles_clean.head()

49 columns (was 49 )


,title,year,album,album_debut,songwriter,songwriter_group,lead_vocal,is_cover,genre,styles,themes,moods,popularity,duration,key,key_name,mode,mode_name,tempo,time_signature,valence,danceability,energy,loudness,acousticness,instrumentalness,liveness,speechiness,other_releases,single_a_side,single_b_side,single_certification,covered_by,billboard_top50,chart_position_uk_wikipedia,chart_position_us_wikipedia,highest_position_the_guardian,weeks_on_chart_in_uk_the_guardian,weeks_at_no1_in_uk_the_guardian,highest_position_billboard,weeks_at_no1_billboard,top_50_billboard,top_50_ultimate_classic_rock,top_50_rolling_stone,top_50_nme,top_50_top50songsorg,top_50_usa_today_2017,top_50_vulture_by_bill_wyman,uri
0,12-bar Original,1965,Anthology 2,Anthology 2,"Lennon, McCartney, Harrison and Starkey",Whole Band,None (instrumental),False,Blues,,,Passionate,31,175,9,A,1,major,122.678,4,0.566,0.600,0.535,-8.178,0.00154,0.102000,0.1230,0.0298,<NA>,NaN,NaN,NaN,<NA>,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,spotify:track:2HvTGx5fzFGpHSyRNvXd9T
1,A Day in the Life,1967,Sgt. Pepper's Lonely Hearts Club Band,Sgt. Pepper's Lonely Hearts Club Band,Lennon and McCartney,Lennon & McCartney,Lennon and McCartney,False,"Psychedelic Rock, Art Rock, Pop/Rock","British Psychedelia, Contemporary Pop/Rock, Ea...",,"Ambitious, Complex, Dramatic, Dreamy, Eerie, E...",65,335,4,E,0,minor,163.219,4,0.175,0.364,0.457,-14.162,0.29000,0.000106,0.9220,0.0675,12,Sgt. Pepper's Lonely Hearts Club Band / With a...,A Day in the Life,NaN,27,False,63.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,2.0,18.0,1.0,1.0,spotify:track:0hKRSZhUGEhKU6aNSPBACZ
2,A Hard Day's Night,1964,A Hard Day's Night,UK: A Hard Day's Night US: 1962-1966,Lennon,Lennon,"Lennon, with McCartney",False,"Rock, Electronic, Pop/Rock","British Invasion, Contemporary Pop/Rock, Early...","Everyday Life, In Love, Relationships","Carefree, Cheerful, Explosive, Marching, Optim...",71,152,0,C,1,major,138.514,4,0.797,0.590,0.805,-6.481,0.13700,0.000000,0.0996,0.0371,35,A Hard Day's Night; A Hard Day's Night,Things We Said Today; I Should Have Known Better,RIAA Gold,35,True,1.0,1.0,1.0,13.0,3.0,1.0,2.0,8.0,18.0,11.0,19.0,19.0,15.0,41.0,spotify:track:5J2CHimS7dWYMImCHkEFaJ
3,A Shot of Rhythm and Blues,1963,Live at the BBC,Live at the BBC,Thompson,Cover,Lennon,True,"R&B, Pop/Rock","British Invasion, Merseybeat, Contemporary Pop...","Celebration, Hanging Out, Partying","Amiable/Good-Natured, Innocent, Swaggering, Br...",35,136,2,D,1,major,128.919,4,0.968,0.699,0.752,-9.565,0.87900,0.000237,0.2650,0.0354,<NA>,NaN,NaN,NaN,<NA>,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,spotify:track:5RuE7nwp4TWsRvLXG7Yjy1
4,A Taste of Honey,1963,Please Please Me,UK: Please Please Me US: The Early Beatles,"Scott, Marlow",Cover,McCartney,True,"Pop/Rock, Jazz, Stage and Screen","British Invasion, Early Pop/Rock, Merseybeat, ...",Affection/Fondness,"Earnest, Precious, Refined, Sentimental, Sweet...",51,163,1,C#/Db,0,minor,101.408,3,0.412,0.420,0.372,-11.416,0.69800,0.000000,0.1040,0.0327,29,NaN,NaN,NaN,<NA>,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,spotify:track:7fh53ta3vAOGJMQ4i5tCHe


A final check of the whole DataFrame:

In [35]:
beatles_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 278 entries, 0 to 277
Data columns (total 49 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   title                              278 non-null    object 
 1   year                               276 non-null    Int64  
 2   album                              278 non-null    str    
 3   album_debut                        278 non-null    str    
 4   songwriter                         278 non-null    str    
 5   songwriter_group                   278 non-null    str    
 6   lead_vocal                         278 non-null    str    
 7   is_cover                           278 non-null    bool   
 8   genre                              278 non-null    str    
 9   styles                             278 non-null    str    
 10  themes                             278 non-null    str    
 11  moods                              278 non-null    str    
 12  popul

### Why Pickle?

A CSV stores everything as text, so when you reload it Pandas must guess the data types all over again: our `Int64` years and our True/False columns would be lost. A **Pickle** (`.pkl`) saves the DataFrame *exactly* as it is in memory, data types included.

> Pickles are for your own work. Never open a pickle from a source you don't trust, and use CSV when sharing data with people who don't use Python.

In [36]:
beatles_clean.to_pickle('beatles_clean.pkl')

# check that it reloads correctly
pd.read_pickle('beatles_clean.pkl').shape

(278, 49)

## 10. Best Practice: One Reproducible Cleaning Function

It can be tempting to fix errors by hand in the spreadsheet. But a key principle of data science is **reproducibility**: anyone should be able to start with the original data and get your result by running your code.

Once you have worked out your cleaning steps (as we did above, one at a time, checking as we went), you can collect them in a single function. Here is a condensed version of everything in this notebook:

In [37]:
def preprocess(csv_path='Beatles_Belgrade_Complete.csv'):
    df = pd.read_csv(csv_path)
    df.columns = [clean_column_name(col) for col in df.columns]

    # missing data
    df = df.dropna(subset=['genre'])
    df['lead_vocal'] = df['lead_vocal'].fillna('None (instrumental)')
    df['album'] = df['album'].fillna('Unreleased')
    df['album_debut'] = df['album_debut'].fillna('Unreleased')
    for col in ['styles', 'themes', 'moods']:
        df[col] = df[col].fillna('')
    df['billboard_top50'] = df['top_50_billboard'].notna()
    df['is_cover'] = df['cover'] == 'Y'
    df = df.drop(columns=['cover'])

    # inconsistent values
    df['year'] = df['year'].str.split('/').str[-1]
    df['title'] = df['title'].str.split(' - ').str[0].str.strip()
    df['styles'] = (df['styles'].str.replace('. ', ', ', regex=False)
                                .str.replace('Prog-RockHard Rock', 'Prog-Rock, Hard Rock', regex=False)
                                .str.replace(r'Pop/Roc,\s*k', 'Pop/Rock', regex=True))
    df['genre'] = df['genre'].str.replace('FolkPop/Rock', 'Folk, Pop/Rock', regex=False)
    df['moods'] = df['moods'].str.split().str.join(', ')
    df['album'] = df['album'].replace(album_fixes)
    df['key_name'] = df['key'].map(key_names)
    df['mode_name'] = df['mode'].map(mode_names)
    for col in ['genre', 'styles', 'themes', 'moods']:
        df[col] = df[col].apply(clean_label_list)
    df['songwriter_group'] = df['songwriter'].apply(songwriter_group)

    # data types
    df['year'] = pd.to_numeric(df['year']).astype('Int64')
    for col in int_cols:
        df[col] = df[col].astype('Int64')

    return df[beatles_clean.columns].reset_index(drop=True)

# the function reproduces our step-by-step result exactly
preprocess().equals(beatles_clean)

True

### Summary of our cleaning decisions

| Problem | Decision |
|---|---|
| 7 Spotify-only outtakes with no metadata | dropped |
| Missing `lead_vocal` (all instrumentals) | filled with `'None (instrumental)'` |
| Missing `album`, `album_debut` | filled with `'Unreleased'` |
| Missing `styles`, `themes`, `moods` | filled with `''` (empty string) |
| Missing chart positions | **kept** as `NaN` (= did not chart); added `billboard_top50` True/False |
| `cover` = `'Y'` or `NaN` | converted to True/False `is_cover` |
| Two-year entries (`1977/1994`) | kept the Beatles' recording year |
| Version suffixes in titles | removed |
| Album name variants | standardized with a dictionary |
| Label typos, capitalization, stray `[` | fixed with a cleaning function |
| Moods separated by spaces | converted to commas |
| Detailed songwriter credits | added simplified `songwriter_group` |
| Wrong data types | `year` and whole-number columns → `Int64` |

**Next:** in **Notebook B** we load `beatles_clean.pkl` and make the data **tidy**.

| **A. Clean Data** | B. Tidy Data | C. Find, Filter, Group | D. Charts and Graphs |
|---|---|---|---|